## 1. Setup and decision rules

Scores: `score_K2 = -K2_lines / sampling_interval`, `score_DET = DET`, `score_PE = -PE`. Stage 1 passes only if K2 wins at least 3 usable primary families by leave-one-family-out Spearman, and its mean LOFO Spearman beats DET and PE by at least 0.05. A family with K2 coverage below 80% is not usable. Stage 2 passes only if K2 has the highest mean family Spearman in at least 8 of 9 conditions and no included condition mean is below 0.50. Stage 3 alerts are: GARCH signed-return false positive, surrogate contrast not tracked, periodic/tie pathology.

In [1]:
import hashlib
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, str(Path("../../src").resolve()))
from recplay.embedding import embed
from recplay.recurrence import euclidean_distances, threshold_by_rate
from recplay.rqa import rqa
from recplay.surrogates import iaaft
from recplay.windowed import beta_distance

RESULT_DIR = Path("../../results/tables")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
SEEDS = (0, 1, 2)
THEILER = 1
K_NEIGHBOURS = 10
N_QUERY = 400
PRIMARY_FAMILIES = ["logistic", "lorenz", "ar1", "garch_abs"]
MEASURE_SCORES = {"K2_lines": "score_K2", "DET": "score_DET", "PE": "score_PE"}


def case_rng(key):
    return np.random.default_rng(int(hashlib.sha256(key.encode()).hexdigest()[:8], 16))


def logistic_series(r, n, rng, burn=1000):
    x, out = rng.uniform(0.2, 0.8), np.empty(n)
    for i in range(burn + n):
        x = r * x * (1 - x)
        if i >= burn:
            out[i - burn] = x
    return out


def _rk4(f, s, dt, n, sub=5):
    out = np.empty((n, len(s)))
    h = dt / sub
    for i in range(n):
        for _ in range(sub):
            k1 = f(s)
            k2 = f(s + h / 2 * k1)
            k3 = f(s + h / 2 * k2)
            k4 = f(s + h * k3)
            s = s + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        out[i] = s
    return out


def lorenz_series(n, rng, dt=0.01, burn=2000):
    f = lambda s: np.array([10 * (s[1] - s[0]), s[0] * (28 - s[2]) - s[1], s[0] * s[1] - 8 / 3 * s[2]])
    s = _rk4(f, rng.normal(0, 1, 3) + [1, 1, 20], dt, burn + n, sub=5)[burn:]
    return s[:, 0]


def ar1_series(phi, n, rng, burn=500):
    sd = np.sqrt(max(1 - phi**2, 0.0))
    e, x = sd * rng.standard_normal(n + burn), np.zeros(n + burn)
    for t in range(1, n + burn):
        x[t] = phi * x[t - 1] + e[t]
    return x[burn:]


def garch_series(persistence, n, rng, alpha=0.08, burn=2000):
    beta = persistence - alpha
    omega = 1 - persistence
    z, r = rng.standard_normal(n + burn), np.empty(n + burn)
    s2 = 1.0
    for t in range(n + burn):
        r[t] = np.sqrt(s2) * z[t]
        s2 = omega + alpha * r[t] ** 2 + beta * s2
    return r[burn:]


_garch_cache = {}


def garch_oracle_skill(persistence, alpha=0.08, horizon=40, n=200_000, seed=0):
    key = (persistence, alpha, horizon, n, seed)
    if key in _garch_cache:
        return _garch_cache[key]
    rng = np.random.default_rng(seed)
    beta, omega, burn = persistence - alpha, 1 - persistence, 2000
    z, r, s2 = rng.standard_normal(n + burn), np.empty(n + burn), np.empty(n + burn + 1)
    s2[0] = 1.0
    for t in range(n + burn):
        r[t] = np.sqrt(s2[t]) * z[t]
        s2[t + 1] = omega + alpha * r[t] ** 2 + beta * s2[t]
    r2, s2 = r[burn:] ** 2, s2[burn:]
    skill = []
    for h in range(1, horizon + 1):
        pred = 1 + persistence ** (h - 1) * (s2[1:len(r2) - h + 1] - 1)
        skill.append(1 - np.mean((r2[h:] - pred) ** 2) / r2.var())
    _garch_cache[key] = np.array(skill)
    return _garch_cache[key]


def lead_steps_for(lead_times, sampling_interval):
    q = np.asarray(lead_times, float) / sampling_interval
    assert np.allclose(q, np.round(q)), (lead_times, sampling_interval, q)
    return np.round(q).astype(int)


def analogue_skill_at_leads(train, test, m, tau, lead_steps, k=10, n_query=400):
    train, test = np.asarray(train, float), np.asarray(test, float)
    lead_steps = np.asarray(lead_steps, int)
    max_lead = int(lead_steps.max())
    lib_states_all = embed(train, m, tau)
    lib_states = lib_states_all[:len(lib_states_all) - max_lead]
    lib_t0 = np.arange(len(lib_states)) + (m - 1) * tau
    q_states_all = embed(test, m, tau)
    q_states_valid = q_states_all[:len(q_states_all) - max_lead]
    q_idx = np.linspace(0, len(q_states_valid) - 1, min(n_query, len(q_states_valid))).astype(int)
    q_states = q_states_valid[q_idx]
    q_t0 = q_idx + (m - 1) * tau
    d = np.sqrt(((q_states[:, None, :] - lib_states[None, :, :]) ** 2).sum(-1))
    nn = np.argpartition(d, k, axis=1)[:, :k]
    var = np.var(test)
    skill = []
    for lead in lead_steps:
        pred = train[lib_t0[nn] + int(lead)].mean(1)
        actual = test[q_t0 + int(lead)]
        skill.append(1 - np.mean((pred - actual) ** 2) / var)
    return np.array(skill)


def skill_summary(skill_by_lead, lead_times):
    skill = np.asarray(skill_by_lead, float)
    times = np.asarray(lead_times, float)
    below = np.flatnonzero(skill < 0.5)
    return {
        "S_time": float(np.clip(skill, 0, None).mean()),
        "H50_time": float(times[below[0]]) if len(below) else float(times.max()),
        "skill_first": float(skill[0]),
    }


def permutation_entropy(x, order=4, delay=1):
    x = np.asarray(x, float)
    n = len(x) - (order - 1) * delay
    patterns = np.argsort(np.stack([x[k * delay:k * delay + n] for k in range(order)], 1), 1)
    _, counts = np.unique(patterns, axis=0, return_counts=True)
    p = counts / counts.sum()
    from math import factorial
    return float(-(p * np.log(p)).sum() / np.log(factorial(order)))


def measure_window(x, m, tau, rr=0.05, distance="euclidean", beta=None):
    x = np.asarray(x, float)
    sd = x.std()
    keys = ["RR", "DET", "L", "ENTR", "LAM", "DET3_DET2", "K2_lines"]
    if not np.isfinite(sd) or sd == 0:
        out = {k: np.nan for k in keys}
        out.update({"PE": np.nan, "K2_missing": True})
        return out
    z = (x - x.mean()) / sd
    if distance == "euclidean":
        d = euclidean_distances(embed(z, m, tau))
    elif distance == "beta":
        d = beta_distance(z, m, tau, beta)
    else:
        raise ValueError(distance)
    out = rqa(threshold_by_rate(d, rr, THEILER))
    out["PE"] = permutation_entropy(z)
    out["K2_missing"] = bool(pd.isna(out["K2_lines"]))
    return out


def spearman_value(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3 or np.unique(x[ok]).size < 2 or np.unique(y[ok]).size < 2:
        return np.nan
    return float(spearmanr(x[ok], y[ok]).statistic)


def acf1(x):
    x = np.asarray(x, float)
    return float(np.corrcoef(x[:-1], x[1:])[0, 1]) if x.std() > 0 else np.nan


assert np.array_equal(lead_steps_for(np.arange(0.2, 2.01, 0.2), 0.05), np.arange(4, 41, 4))
assert np.array_equal(lead_steps_for(np.arange(0.2, 2.01, 0.2), 0.10), np.arange(2, 21, 2))
smoke = measure_window(case_rng("smoke-normal").standard_normal(200), 1, 1)
assert np.isfinite(smoke["DET"]) and np.isfinite(smoke["PE"]) and (np.isfinite(smoke["K2_lines"]) or smoke["K2_missing"])
constant = measure_window(np.ones(200), 1, 1)
assert all(pd.isna(constant[k]) for k in ["RR", "DET", "L", "ENTR", "LAM", "DET3_DET2", "K2_lines", "PE"]) and constant["K2_missing"]
white_k2 = [measure_window(case_rng(f"sanity-white-noise|{i}").standard_normal(1000), 1, 1)["K2_lines"] for i in range(3)]
print(f"white-noise K2_lines mean={np.nanmean(white_k2):.3f} values={np.round(white_k2, 3)}")
assert 2.90 <= np.nanmean(white_k2) <= 3.10

white-noise K2_lines mean=3.023 values=[3.031 3.057 2.981]


## 2. Sampling test

Primary families: logistic, Lorenz, AR(1), GARCH absolute returns. Skill is compared at common physical lead times; K2 is divided by the sampling interval.

In [2]:
stage1_cases = []
stage1_rows = []
lead_logistic = np.arange(4, 41, 4, dtype=float)
lead_time = np.round(np.arange(0.2, 2.01, 0.2), 10)
t_stage1 = time.time()

# Logistic skip-sampling
for seed in SEEDS:
    for r in np.linspace(3.5, 4.0, 13):
        for interval in (1, 2, 4):
            original = logistic_series(float(r), (6000 + 1040) * interval, case_rng(f"logistic|{r:.3f}|{seed}"))
            observed = original[::interval]
            train, test = observed[:6000], observed[6000:7040]
            window = test[:1000]
            skill = analogue_skill_at_leads(train, test, 1, 1, lead_steps_for(lead_logistic, interval), K_NEIGHBOURS, N_QUERY)
            summ = skill_summary(skill, lead_logistic)
            meas = measure_window(window, 1, 1, 0.05)
            row = {"stage": "stage1", "family": "logistic", "param": float(r), "seed": seed,
                   "sampling_interval": float(interval), "m": 1, "tau_steps": 1,
                   "window_points": len(window), "window_duration": float(len(window) * interval),
                   "lead_times": ";".join(map(str, lead_logistic)), **summ, "target": "analogue",
                   "ACF1": acf1(window), **meas}
            row.update({"score_K2": -row["K2_lines"] / interval, "score_DET": row["DET"], "score_PE": -row["PE"]})
            stage1_rows.append(row)
            stage1_cases.append({**row, "train": train, "test": test, "window": window})

# Lorenz decimation from one clean high-resolution trajectory per seed
n_high = int(round(350 / 0.01))
for seed in SEEDS:
    clean = lorenz_series(n_high, case_rng(f"lorenz|{seed}"), dt=0.01)
    for noise in (0, 0.02, 0.05, 0.1, 0.2, 0.5):
        noisy = clean + noise * clean.std() * case_rng(f"lorenz-noise|{noise}|{seed}").standard_normal(len(clean))
        for interval, q, tau_steps in ((0.02, 2, 5), (0.05, 5, 2), (0.10, 10, 1)):
            observed = noisy[::q]
            n_train, n_test = int(round(300 / interval)), int(round(50 / interval))
            train, test = observed[:n_train], observed[n_train:n_train + n_test]
            window = test.copy()
            skill = analogue_skill_at_leads(train, test, 3, tau_steps, lead_steps_for(lead_time, interval), K_NEIGHBOURS, N_QUERY)
            summ = skill_summary(skill, lead_time)
            meas = measure_window(window, 3, tau_steps, 0.05)
            row = {"stage": "stage1", "family": "lorenz", "param": float(noise), "seed": seed,
                   "sampling_interval": float(interval), "m": 3, "tau_steps": tau_steps,
                   "window_points": len(window), "window_duration": 50.0,
                   "lead_times": ";".join(map(str, lead_time)), **summ, "target": "analogue",
                   "ACF1": acf1(window), **meas}
            row.update({"score_K2": -row["K2_lines"] / interval, "score_DET": row["DET"], "score_PE": -row["PE"]})
            stage1_rows.append(row)
            stage1_cases.append({**row, "train": train, "test": test, "window": window})

# Sampled AR(1), exact skill
for seed in SEEDS:
    for kappa in (0.05, 0.1, 0.2, 0.5, 1.0, 2.0):
        for interval in (0.02, 0.05, 0.10):
            phi = np.exp(-kappa * interval)
            x = ar1_series(phi, 7040, case_rng(f"ar1|{kappa}|{interval}|{seed}"))
            train, test = x[:6000], x[6000:7040]
            window = test[:1000]
            summ = skill_summary(np.exp(-2 * kappa * lead_time), lead_time)
            meas = measure_window(window, 1, 1, 0.05)
            row = {"stage": "stage1", "family": "ar1", "param": float(kappa), "seed": seed,
                   "sampling_interval": float(interval), "m": 1, "tau_steps": 1,
                   "window_points": len(window), "window_duration": float(len(window) * interval),
                   "lead_times": ";".join(map(str, lead_time)), **summ, "target": "theory",
                   "ACF1": acf1(window), **meas}
            row.update({"score_K2": -row["K2_lines"] / interval, "score_DET": row["DET"], "score_PE": -row["PE"]})
            stage1_rows.append(row)
            stage1_cases.append({**row, "train": train, "test": test, "window": window})

# GARCH absolute returns, true-model volatility target
for seed in SEEDS:
    for persistence in (0.5, 0.8, 0.9, 0.95, 0.98, 0.995):
        x = garch_series(persistence, 7040, case_rng(f"garch|{persistence}|{seed}"))
        train, test = x[:6000], x[6000:7040]
        window = np.abs(test[:1000])
        oracle = garch_oracle_skill(persistence)
        summ = skill_summary(oracle[(lead_logistic - 1).astype(int)], lead_logistic)
        meas = measure_window(window, 2, 1, 0.05)
        row = {"stage": "stage1", "family": "garch_abs", "param": float(persistence), "seed": seed,
               "sampling_interval": 1.0, "m": 2, "tau_steps": 1,
               "window_points": len(window), "window_duration": float(len(window)),
               "lead_times": ";".join(map(str, lead_logistic)), **summ, "target": "garch_oracle",
               "ACF1": acf1(window), **meas}
        row.update({"score_K2": -row["K2_lines"], "score_DET": row["DET"], "score_PE": -row["PE"]})
        stage1_rows.append(row)
        stage1_cases.append({**row, "train": np.abs(train), "test": np.abs(test), "window": window})

stage1 = pd.DataFrame(stage1_rows)
stage1_columns = ["stage", "family", "param", "seed", "sampling_interval", "m", "tau_steps", "window_points",
                  "window_duration", "lead_times", "S_time", "H50_time", "skill_first", "target", "RR", "DET", "L",
                  "ENTR", "LAM", "DET3_DET2", "K2_lines", "K2_missing", "PE", "ACF1", "score_K2", "score_DET", "score_PE"]
stage1 = stage1[stage1_columns]
stage1.to_csv(RESULT_DIR / "k2_sampling_cases.csv", index=False)
print(f"Stage 1: {len(stage1)} cases in {time.time() - t_stage1:.0f} s")
stage1.groupby("family")["K2_lines"].agg(["count", "mean"]).round(3)

Stage 1: 243 cases in 103 s


,count,mean
family,,
ar1,54,1.513
garch_abs,18,1.443
logistic,117,0.626
lorenz,54,0.415


In [3]:
def build_summary(df):
    rows = []
    for family in PRIMARY_FAMILIES:
        g = df[df.family == family]
        coverage = float((~g["K2_lines"].isna()).mean())
        lofo = {}
        rmse = {}
        for measure, score in MEASURE_SCORES.items():
            train_all = df[(df.family != family)]
            ok_train = train_all[score].notna() & train_all["S_time"].notna()
            coef = np.linalg.lstsq(np.c_[np.ones(ok_train.sum()), train_all.loc[ok_train, score]], train_all.loc[ok_train, "S_time"], rcond=None)[0]
            pred = coef[0] + coef[1] * g[score]
            lofo[measure] = spearman_value(pred, g["S_time"])
            ok = np.isfinite(pred) & np.isfinite(g["S_time"])
            rmse[measure] = float(np.sqrt(np.mean((pred[ok] - g.loc[ok, "S_time"]) ** 2))) if ok.any() else np.nan
        winners = [m for m in MEASURE_SCORES if np.isfinite(lofo[m]) and all(lofo[m] > lofo[o] for o in MEASURE_SCORES if o != m and np.isfinite(lofo[o]))]
        for measure, score in MEASURE_SCORES.items():
            rows.append({"scope": "family", "family": family, "sampling_interval": np.nan, "measure": measure,
                         "n_cases": len(g), "n_seeds": g.seed.nunique(), "K2_coverage": coverage,
                         "direct_spearman": spearman_value(g[score], g["S_time"]), "lofo_spearman": lofo[measure],
                         "lofo_RMSE": rmse[measure], "wins_family": bool(measure == "K2_lines" and winners == ["K2_lines"])})
        for interval, gg in g.groupby("sampling_interval"):
            for measure, score in MEASURE_SCORES.items():
                rows.append({"scope": "condition", "family": family, "sampling_interval": interval, "measure": measure,
                             "n_cases": len(gg), "n_seeds": gg.seed.nunique(), "K2_coverage": float((~gg["K2_lines"].isna()).mean()),
                             "direct_spearman": spearman_value(gg[score], gg["S_time"]), "lofo_spearman": np.nan,
                             "lofo_RMSE": np.nan, "wins_family": False})
    return pd.DataFrame(rows)


def gate_from_summary(summary):
    fam = summary[summary.scope == "family"]
    k2 = fam[fam.measure == "K2_lines"].set_index("family")
    usable = [f for f in PRIMARY_FAMILIES if k2.loc[f, "K2_coverage"] >= 0.8 and np.isfinite(k2.loc[f, "lofo_spearman"])]
    means = {}
    for measure in MEASURE_SCORES:
        vals = fam[(fam.measure == measure) & fam.family.isin(usable)]["lofo_spearman"]
        means[measure] = float(vals.mean()) if len(vals) else np.nan
    wins = int(k2.loc[usable, "wins_family"].sum()) if usable else 0
    passed = len(usable) >= 3 and wins >= 3 and means["K2_lines"] - means["DET"] >= 0.05 and means["K2_lines"] - means["PE"] >= 0.05
    return passed, usable, wins, means


stage1_summary = build_summary(stage1)
stage1_summary.to_csv(RESULT_DIR / "k2_sampling_summary.csv", index=False)
stage1_filled = stage1.copy()
stage1_filled["K2_lines"] = stage1_filled["K2_lines"].fillna(-np.log(stage1_filled["RR"]))
stage1_filled["score_K2"] = -stage1_filled["K2_lines"] / stage1_filled["sampling_interval"]
stage1_sensitivity = build_summary(stage1_filled)
stage1_sensitivity.to_csv(RESULT_DIR / "k2_sampling_sensitivity.csv", index=False)
stage1_gate_pass, stage1_usable, stage1_k2_wins, stage1_means = gate_from_summary(stage1_summary)
stage1_diff_det = stage1_means["K2_lines"] - stage1_means["DET"]
stage1_diff_pe = stage1_means["K2_lines"] - stage1_means["PE"]
print(f"Stage 1 gate pass={stage1_gate_pass}; usable={stage1_usable}; K2 wins={stage1_k2_wins}")
print(f"Mean LOFO={stage1_means}; K2-DET={stage1_diff_det:.3f}; K2-PE={stage1_diff_pe:.3f}")
stage1_summary[stage1_summary.scope == "family"].round(3)

Stage 1 gate pass=False; usable=['logistic', 'lorenz', 'ar1', 'garch_abs']; K2 wins=1
Mean LOFO={'K2_lines': 0.6560496102761996, 'DET': 0.8143633203024062, 'PE': 0.5359334680311477}; K2-DET=-0.158; K2-PE=0.120


,scope,family,sampling_interval,measure,n_cases,n_seeds,K2_coverage,direct_spearman,lofo_spearman,lofo_RMSE,wins_family
0,family,logistic,NaN,K2_lines,117,3,1.0,0.956,0.956,0.358,True
1,family,logistic,NaN,DET,117,3,1.0,0.788,0.788,0.238,False
2,family,logistic,NaN,PE,117,3,1.0,0.808,0.809,0.182,False
12,family,lorenz,NaN,K2_lines,54,3,1.0,0.706,0.706,0.258,False
13,family,lorenz,NaN,DET,54,3,1.0,0.938,0.938,0.291,False
14,family,lorenz,NaN,PE,54,3,1.0,0.923,0.923,0.171,False
24,family,ar1,NaN,K2_lines,54,3,1.0,0.407,0.407,0.412,False
25,family,ar1,NaN,DET,54,3,1.0,0.833,0.833,0.383,False
26,family,ar1,NaN,PE,54,3,1.0,0.271,0.271,0.414,False
36,family,garch_abs,NaN,K2_lines,18,3,1.0,0.555,0.555,0.530,False


## 3. Robustness test

Nine conditions: window length, recurrence rate, and beta distance. Runs only if Stage 1 passes.

In [4]:
stage2 = pd.DataFrame()
stage2_summary = pd.DataFrame()
stage2_pass = False
stage2_wins = 0
stage2_fraction = 0.0
stage2_min_mean = np.nan
stage2_excluded = []
stage2_condition_means = pd.DataFrame()

if not stage1_gate_pass:
    print("Stage 2 skipped: Stage 1 gate failed")
else:
    t_stage2 = time.time()
    baseline_cases = [c for c in stage1_cases if
                      (c["family"] == "logistic" and c["sampling_interval"] == 1.0) or
                      (c["family"] == "lorenz" and c["sampling_interval"] == 0.05) or
                      (c["family"] == "ar1" and c["sampling_interval"] == 0.05) or
                      (c["family"] == "garch_abs" and c["sampling_interval"] == 1.0)]
    conditions = [
        ("W250_RR0.05_euclidean", 250, 0.05, "euclidean", np.nan),
        ("W500_RR0.05_euclidean", 500, 0.05, "euclidean", np.nan),
        ("W1000_RR0.05_euclidean", 1000, 0.05, "euclidean", np.nan),
        ("W1000_RR0.02_euclidean", 1000, 0.02, "euclidean", np.nan),
        ("W1000_RR0.10_euclidean", 1000, 0.10, "euclidean", np.nan),
        ("W1000_RR0.20_euclidean", 1000, 0.20, "euclidean", np.nan),
        ("W1000_RR0.05_beta-1", 1000, 0.05, "beta", -1.0),
        ("W1000_RR0.05_beta1", 1000, 0.05, "beta", 1.0),
        ("W1000_RR0.05_beta2", 1000, 0.05, "beta", 2.0),
    ]
    rows2 = []
    for c in baseline_cases:
        window = np.asarray(c["window"], float)
        z = (window - window.mean()) / window.std()
        states = embed(z, int(c["m"]), int(c["tau_steps"]))
        d_euc = euclidean_distances(states)
        d_beta = {}
        for b in (-1.0, 1.0, 2.0):
            try:
                d_beta[b] = beta_distance(z, int(c["m"]), int(c["tau_steps"]), b)
            except Exception as exc:
                print(f"beta {b} failed for {c['family']} seed {c['seed']}: {exc}")
                d_beta[b] = None
        for condition_id, W, rr, distance, beta in conditions:
            try:
                if distance == "euclidean":
                    n_states = W - (int(c["m"]) - 1) * int(c["tau_steps"])
                    d = d_euc[:n_states, :n_states]
                else:
                    d = d_beta[float(beta)]
                    if d is None:
                        raise ValueError("beta distance failed")
                meas = rqa(threshold_by_rate(d, rr, THEILER))
                meas["PE"] = permutation_entropy(window[:W])
                meas["K2_missing"] = bool(pd.isna(meas["K2_lines"]))
            except Exception as exc:
                print(f"{condition_id} failed for {c['family']} seed {c['seed']}: {exc}")
                meas = {"RR": np.nan, "DET": np.nan, "L": np.nan, "ENTR": np.nan, "LAM": np.nan,
                        "DET3_DET2": np.nan, "K2_lines": np.nan, "PE": permutation_entropy(window[:W]), "K2_missing": True}
            row = {k: c[k] for k in ["family", "param", "seed", "sampling_interval", "m", "tau_steps", "lead_times",
                                     "S_time", "H50_time", "skill_first", "target"]}
            row.update({"stage": "stage2", "condition_id": condition_id, "window_points": W,
                        "window_duration": float(W * c["sampling_interval"]), "distance": distance, "beta": beta,
                        "ACF1": acf1(window[:W]), **meas})
            row.update({"score_K2": -row["K2_lines"] / c["sampling_interval"] if pd.notna(row["K2_lines"]) else np.nan,
                        "score_DET": row["DET"], "score_PE": -row["PE"] if pd.notna(row["PE"]) else np.nan})
            rows2.append(row)
    stage2 = pd.DataFrame(rows2)
    stage2.to_csv(RESULT_DIR / "k2_robustness_cases.csv", index=False)
    srows = []
    for (condition_id, family), g in stage2.groupby(["condition_id", "family"]):
        for measure, score in MEASURE_SCORES.items():
            srows.append({"condition_id": condition_id, "family": family, "measure": measure,
                          "spearman": spearman_value(g[score], g["S_time"]),
                          "K2_coverage": float((~g["K2_lines"].isna()).mean()), "n_cases": len(g)})
    stage2_summary = pd.DataFrame(srows)
    stage2_summary.to_csv(RESULT_DIR / "k2_robustness_summary.csv", index=False)
    cond = stage2_summary.groupby(["condition_id", "measure"])["spearman"].mean().unstack()
    coverage_bad = stage2_summary[stage2_summary.K2_coverage < 0.8].condition_id.unique().tolist()
    stage2_excluded = coverage_bad
    included = [c for c in cond.index if c not in coverage_bad]
    stage2_min_mean = float(cond.loc[included, "K2_lines"].min()) if included else np.nan
    stage2_wins = int(sum(cond.loc[c, "K2_lines"] > cond.loc[c, "DET"] and cond.loc[c, "K2_lines"] > cond.loc[c, "PE"] for c in cond.index)) if len(cond) else 0
    stage2_fraction = stage2_wins / 9
    stage2_condition_means = cond
    stage2_pass = stage2_wins >= 8 and stage2_min_mean >= 0.50
    print(f"Stage 2: {len(stage2)} rows in {time.time() - t_stage2:.0f} s; wins={stage2_wins}/9; min included K2 mean={stage2_min_mean:.3f}; excluded={stage2_excluded}; pass={stage2_pass}")
    stage2_condition_means.round(3)

Stage 2 skipped: Stage 1 gate failed


## 4. Misleading cases

Runs only if Stages 1 and 2 pass. Panels: GARCH forms, IAAFT surrogates, periodic windows.

In [5]:
stage3 = pd.DataFrame()
stage3_summary = pd.DataFrame()
garch_false_positive = False
surrogate_alert = False
surrogate_label = "not run"
periodic_pathology = False

if not (stage1_gate_pass and stage2_pass):
    print("Stage 3 skipped by gate")
else:
    t_stage3 = time.time()
    rows3 = []
    summary3 = []
    # Panel A: GARCH forms from the same realization
    for seed in SEEDS:
        for persistence in (0.5, 0.8, 0.9, 0.95, 0.98, 0.995):
            x = garch_series(persistence, 7040, case_rng(f"garch|{persistence}|{seed}"))
            test = x[6000:7040]
            oracle = garch_oracle_skill(persistence)
            s_oracle = skill_summary(oracle[(lead_logistic - 1).astype(int)], lead_logistic)["S_time"]
            for form, observable, s_time in (("garch_returns", test[:1000], 0.0),
                                             ("garch_abs", np.abs(test[:1000]), s_oracle),
                                             ("garch_r2", test[:1000] ** 2, s_oracle)):
                meas = measure_window(observable, 2, 1, 0.05)
                row = {"panel": "garch", "family": "garch", "form": form, "param": float(persistence), "seed": seed,
                       "pair_id": f"garch:{persistence}:{seed}", "S_time": float(s_time), **meas}
                row.update({"score_K2": -row["K2_lines"] if pd.notna(row["K2_lines"]) else np.nan,
                            "score_DET": row["DET"], "score_PE": -row["PE"] if pd.notna(row["PE"]) else np.nan})
                rows3.append(row)
    panel_a = pd.DataFrame([r for r in rows3 if r["panel"] == "garch"])
    for form, g in panel_a.groupby("form"):
        avg = g.groupby("param")[["score_K2", "DET", "PE"]].mean(numeric_only=True)
        vals = {"score_K2": spearman_value(avg.index, avg["score_K2"]), "DET": spearman_value(avg.index, avg["DET"]),
                "score_PE": spearman_value(avg.index, -avg["PE"])}
        for measure, val in vals.items():
            summary3.append({"panel": "garch", "criterion": f"{form} persistence vs {measure}", "observed": val,
                             "threshold": 0.50, "passed": bool(val >= 0.50) if np.isfinite(val) else False, "alert": False})
    signed_avg = panel_a[panel_a.form == "garch_returns"].groupby("param")["score_K2"].mean()
    garch_signed_rho = spearman_value(signed_avg.index, signed_avg.values)
    garch_false_positive = bool(np.isfinite(garch_signed_rho) and garch_signed_rho >= 0.50)
    summary3.append({"panel": "garch", "criterion": "garch_false_positive", "observed": garch_signed_rho,
                     "threshold": 0.50, "passed": not garch_false_positive, "alert": garch_false_positive})

    # Panel B: IAAFT surrogates, independently for train and test
    pairs = []
    for seed in SEEDS:
        for r in (3.8, 3.9, 4.0):
            pairs.append(("logistic", float(r), seed))
        pairs.append(("lorenz", 0.0, seed))
    for family, param, seed in pairs:
        pair_id = f"{family}:{param}:{seed}"
        if family == "logistic":
            x = logistic_series(param, 7040, case_rng(f"logistic|{param:.3f}|{seed}"))
            train, test, m, tau = x[:6000], x[6000:7040], 1, 1
            leads, times = lead_steps_for(lead_logistic, 1), lead_logistic
        else:
            clean = lorenz_series(n_high, case_rng(f"lorenz|{seed}"), dt=0.01)
            observed = clean[::5]
            train, test, m, tau = observed[:6000], observed[6000:7000], 3, 2
            leads, times = lead_steps_for(lead_time, 0.05), lead_time
        variants = (("original", train, test),
                    ("surrogate", iaaft(train, case_rng(f"iaaft-train|{pair_id}")), iaaft(test, case_rng(f"iaaft-test|{pair_id}"))))
        for form, tr, te in variants:
            skill = analogue_skill_at_leads(tr, te, m, tau, leads, K_NEIGHBOURS, N_QUERY)
            summ = skill_summary(skill, times)
            meas = measure_window(te[:1000], m, tau, 0.05)
            row = {"panel": "surrogate", "family": family, "form": form, "param": float(param), "seed": seed,
                   "pair_id": pair_id, "S_time": summ["S_time"], **meas}
            interval = 1.0 if family == "logistic" else 0.05
            row.update({"score_K2": -row["K2_lines"] / interval if pd.notna(row["K2_lines"]) else np.nan,
                        "score_DET": row["DET"], "score_PE": -row["PE"] if pd.notna(row["PE"]) else np.nan})
            rows3.append(row)
    panel_b = pd.DataFrame([r for r in rows3 if r["panel"] == "surrogate"])
    wide = panel_b.pivot_table(index="pair_id", columns="form", values=["S_time", "score_K2", "DET", "PE"], aggfunc="first")
    diffs = {"S_time": float((wide[("S_time", "original")] - wide[("S_time", "surrogate")]).median()),
             "score_K2": float((wide[("score_K2", "original")] - wide[("score_K2", "surrogate")]).median()),
             "DET": float((wide[("DET", "original")] - wide[("DET", "surrogate")]).median()),
             "score_PE": float((-wide[("PE", "original")] + wide[("PE", "surrogate")]).median())}
    for name, val in diffs.items():
        summary3.append({"panel": "surrogate", "criterion": f"median original-surrogate {name}", "observed": val,
                         "threshold": 0.0, "passed": bool(val > 0), "alert": False})
    if diffs["S_time"] > 0 and diffs["score_K2"] > 0:
        surrogate_alert, surrogate_label = False, "contrast tracked"
    elif diffs["S_time"] > 0:
        surrogate_alert, surrogate_label = True, "surrogate contrast not tracked"
    else:
        surrogate_alert, surrogate_label = False, "target did not create a contrast"
    summary3.append({"panel": "surrogate", "criterion": "surrogate_alert", "observed": surrogate_label,
                     "threshold": "skill falls and K2 falls", "passed": not surrogate_alert, "alert": surrogate_alert})

    # Panel C: periodic windows
    for seed in SEEDS:
        for frequency in (0.05, 0.10):
            for noise in (0, 0.02, 0.05, 0.1, 0.2, 0.5):
                rng = case_rng(f"sine|{frequency}|{noise}|{seed}")
                t = np.arange(7040)
                clean = np.sin(2 * np.pi * frequency * t + rng.uniform(0, 2 * np.pi))
                x = clean + noise * clean.std() * rng.standard_normal(len(t))
                train, test = x[:6000], x[6000:7040]
                times_c = np.arange(1, 41, dtype=float)
                skill = analogue_skill_at_leads(train, test, 1, 1, times_c.astype(int), K_NEIGHBOURS, N_QUERY)
                summ = skill_summary(skill, times_c)
                meas = measure_window(test[:1000], 1, 1, 0.05)
                row = {"panel": "sine", "family": "sine", "form": "sine", "param": float(frequency), "seed": seed,
                       "pair_id": f"sine:{frequency}:{noise}:{seed}", "S_time": summ["S_time"], **meas}
                row.update({"score_K2": -row["K2_lines"] if pd.notna(row["K2_lines"]) else np.nan,
                            "score_DET": row["DET"], "score_PE": -row["PE"] if pd.notna(row["PE"]) else np.nan})
                rows3.append(row)
    panel_c = pd.DataFrame([r for r in rows3 if r["panel"] == "sine"])
    sine_coverage = float((~panel_c["K2_lines"].isna()).mean())
    sine_rhos = {m: spearman_value(panel_c[s], panel_c["S_time"]) for m, s in MEASURE_SCORES.items()}
    for measure, val in sine_rhos.items():
        summary3.append({"panel": "sine", "criterion": f"sine S_time vs {measure}", "observed": val,
                         "threshold": 0.50, "passed": bool(val >= 0.50) if np.isfinite(val) else False, "alert": False})
    summary3.append({"panel": "sine", "criterion": "K2_coverage", "observed": sine_coverage,
                     "threshold": 0.80, "passed": sine_coverage >= 0.80, "alert": sine_coverage < 0.80})
    periodic_pathology = bool(sine_coverage < 0.80 or (sine_rhos["K2_lines"] < 0.50 and (sine_rhos["DET"] >= 0.50 or sine_rhos["PE"] >= 0.50)))
    summary3.append({"panel": "sine", "criterion": "periodic_pathology", "observed": periodic_pathology,
                     "threshold": False, "passed": not periodic_pathology, "alert": periodic_pathology})

    stage3 = pd.DataFrame(rows3)
    stage3_columns = ["panel", "family", "form", "param", "seed", "pair_id", "S_time", "RR", "DET", "L", "ENTR", "LAM",
                      "DET3_DET2", "K2_lines", "K2_missing", "PE", "score_K2", "score_DET", "score_PE"]
    stage3 = stage3[stage3_columns]
    stage3.to_csv(RESULT_DIR / "k2_misleading_cases.csv", index=False)
    stage3_summary = pd.DataFrame(summary3)
    stage3_summary.to_csv(RESULT_DIR / "k2_misleading_summary.csv", index=False)
    print(f"Stage 3: {len(stage3)} rows in {time.time() - t_stage3:.0f} s")
    print(f"garch_false_positive={garch_false_positive}; surrogate_alert={surrogate_alert} ({surrogate_label}); periodic_pathology={periodic_pathology}")
    stage3_summary.round(3)

Stage 3 skipped by gate


## 5. Decision

The verdict is assigned mechanically from the frozen gates.

In [6]:
decision_rows = [
    {"stage": "stage1", "criterion": "K2 family wins", "observed": stage1_k2_wins, "threshold": 3,
     "passed": bool(stage1_k2_wins >= 3), "verdict_effect": "gate"},
    {"stage": "stage1", "criterion": "mean LOFO K2 minus DET", "observed": stage1_diff_det, "threshold": 0.05,
     "passed": bool(stage1_diff_det >= 0.05), "verdict_effect": "gate"},
    {"stage": "stage1", "criterion": "mean LOFO K2 minus PE", "observed": stage1_diff_pe, "threshold": 0.05,
     "passed": bool(stage1_diff_pe >= 0.05), "verdict_effect": "gate"},
]
stage1_family_rows = stage1_summary[stage1_summary.scope == "family"]
for family in PRIMARY_FAMILIES:
    cov = float(stage1_family_rows[(stage1_family_rows.family == family) & (stage1_family_rows.measure == "K2_lines")]["K2_coverage"].iloc[0])
    decision_rows.append({"stage": "stage1", "criterion": f"K2 coverage {family}", "observed": cov, "threshold": 0.80,
                          "passed": bool(cov >= 0.80), "verdict_effect": "usable" if cov >= 0.80 else "not usable"})

if stage1_gate_pass:
    decision_rows += [
        {"stage": "stage2", "criterion": "K2 condition wins", "observed": stage2_fraction, "threshold": 8 / 9,
         "passed": bool(stage2_wins >= 8), "verdict_effect": "gate"},
        {"stage": "stage2", "criterion": "minimum included condition mean K2", "observed": stage2_min_mean, "threshold": 0.50,
         "passed": bool(stage2_min_mean >= 0.50), "verdict_effect": "gate"},
    ]
    for cond_id in stage2_excluded:
        decision_rows.append({"stage": "stage2", "criterion": f"excluded condition {cond_id}", "observed": "K2 coverage < 0.80",
                              "threshold": 0.80, "passed": False, "verdict_effect": "not usable"})
else:
    decision_rows += [
        {"stage": "stage2", "criterion": "K2 condition wins", "observed": "not run", "threshold": 8 / 9,
         "passed": False, "verdict_effect": "Stage 1 gate"},
        {"stage": "stage2", "criterion": "minimum included condition mean K2", "observed": "not run", "threshold": 0.50,
         "passed": False, "verdict_effect": "Stage 1 gate"},
    ]

if stage1_gate_pass and stage2_pass:
    decision_rows += [
        {"stage": "stage3", "criterion": "garch_false_positive", "observed": bool(garch_false_positive), "threshold": False,
         "passed": not garch_false_positive, "verdict_effect": "named limit" if garch_false_positive else "none"},
        {"stage": "stage3", "criterion": "surrogate_alert", "observed": bool(surrogate_alert), "threshold": False,
         "passed": not surrogate_alert, "verdict_effect": "named limit" if surrogate_alert else surrogate_label},
        {"stage": "stage3", "criterion": "periodic_pathology", "observed": bool(periodic_pathology), "threshold": False,
         "passed": not periodic_pathology, "verdict_effect": "named limit" if periodic_pathology else "none"},
    ]
else:
    skip_gate = "Stage 1 gate" if not stage1_gate_pass else "Stage 2 gate"
    for name in ("garch_false_positive", "surrogate_alert", "periodic_pathology"):
        decision_rows.append({"stage": "stage3", "criterion": name, "observed": "not run", "threshold": False,
                              "passed": False, "verdict_effect": skip_gate})

if not stage1_gate_pass:
    final_verdict = "sampling artefact / not supported"
elif not stage2_pass:
    final_verdict = "setting-sensitive"
elif garch_false_positive or surrogate_alert or periodic_pathology:
    final_verdict = "ranking gauge with named limits"
else:
    final_verdict = "ranking gauge supported"

decision = pd.DataFrame(decision_rows, columns=["stage", "criterion", "observed", "threshold", "passed", "verdict_effect"])
decision.to_csv(RESULT_DIR / "k2_decision.csv", index=False)
print(f"FINAL VERDICT: {final_verdict}")
decision

FINAL VERDICT: sampling artefact / not supported


,stage,criterion,observed,threshold,passed,verdict_effect
0,stage1,K2 family wins,1,3,False,gate
1,stage1,mean LOFO K2 minus DET,-0.158314,0.05,False,gate
2,stage1,mean LOFO K2 minus PE,0.120116,0.05,True,gate
3,stage1,K2 coverage logistic,1.0,0.8,True,usable
4,stage1,K2 coverage lorenz,1.0,0.8,True,usable
5,stage1,K2 coverage ar1,1.0,0.8,True,usable
6,stage1,K2 coverage garch_abs,1.0,0.8,True,usable
7,stage2,K2 condition wins,not run,0.888889,False,Stage 1 gate
8,stage2,minimum included condition mean K2,not run,0.5,False,Stage 1 gate
9,stage3,garch_false_positive,not run,False,False,Stage 1 gate


## 6. Findings

**Verdict: sampling artefact / not supported.**

Sanity checks pass: white-noise `K2_lines` mean = 3.023 (expected ≈ 3.00); logistic r = 4 `K2_lines` mean = 0.708 (expected 0.65–0.75). K2 coverage was 100% in every primary family, so no family was marked not usable.

Stage 1 (243 cases, skill per unit time): K2 won only 1 of 4 families by leave-one-family-out Spearman (logistic 0.956). On the pooled families, DET beat K2 on Lorenz (0.938 vs 0.706), AR(1) (0.833 vs 0.407), and GARCH |r| (0.699 vs 0.555). Mean LOFO Spearman: K2 0.656, DET 0.814, PE 0.536. The gate failed: K2 − DET = −0.158 (needed ≥ +0.05); K2 − PE = +0.120 (passed).

Within a single sampling interval K2 ranked well (Lorenz 0.92–0.97, AR(1) 0.82–0.97, logistic 0.97–0.99), but pooling intervals within a family broke the ranking, worst for AR(1). Dividing `K2_lines` by the sampling interval did not make scores comparable across sampling rates.

Stages 2 and 3 were not run, as specified by the Stage 1 gate. The earlier per-step result therefore does not survive scoring per unit time with this sampling correction: as a cross-sampling ranking gauge, `K2_lines` is not supported.
